# Encoded VaR for the Nifty 50

**Final capstone, Group 5.** Georgy S Pulayath, Joshi Bhavin Rasikbhai, Prabhujeet Sahu, Shrenik Nandre. Guide: Dr. Puneet Pasricha (IIT Ropar).

This notebook trains a variational autoencoder on daily Nifty 50 returns and reads 1-day 95% VaR from decoded scenarios. The written method and the reported **AE ratio of 1.04** are in `Encoded VaR.md` and in section 7 of the capstone report.

The file contains **two complete implementations**. They define the same class names (`VariationalAutoencoder`, `EncodedVaR`). Run the first through its `main()`, or restart the kernel and run the second. Running both in one session replaces the first model.

| Implementation | What is distinctive |
| --- | --- |
| First | Rolling 90-day z-score, scenarios drawn from the standard normal prior, VaR destandardised with the current mean and volatility, 10-day test blocks |
| Second | Raw simple returns, custom `train_step`, scenarios drawn around the encoded posterior, 21-day step, breach rule `data_point < -VaR` |

Data file: `data/Nifty50_data.xlsx` (`Dates`, `NIFTY Index`), 4 January 2005 to 25 September 2023. Training window in both `main()` functions: 5 January 2005 through 31 December 2017. Fine-tune and backtest: after 31 December 2017.

The original notebook mounted Google Drive in Colab (`drive.mount('/content/drive')`) and read `/content/drive/MyDrive/MSDSM/Sem_4/FCP/EncodedVaR/Nifty50_data.xlsx`. That mount is unnecessary here.


In [ ]:
# Shared dataset locator.
# The workbook and the two-year CSV live in the project data/ folder.
# Running this notebook from a method folder (Historical, GARCH, ...) or from
# the project root both resolve. Machine-specific paths from the original
# Colab / local drives are no longer required.
from pathlib import Path

def find_nifty_file(filename="Nifty50_data.xlsx"):
    """Return the first existing copy of a file under data/ or the working directory."""
    here = Path.cwd().resolve()
    candidates = [
        here / "data" / filename,
        here.parent / "data" / filename,
        here / filename,
    ]
    for path in candidates:
        if path.exists():
            return str(path.resolve())
    raise FileNotFoundError(
        f"Could not find {filename}. Open the notebook from the project folder "
        "or from its method folder so that data/ is visible."
    )

NIFTY_PATH = find_nifty_file("Nifty50_data.xlsx")
print("Using dataset:", NIFTY_PATH)

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
import tensorflow.keras.backend as K
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow.keras as keras

# SGD is imported because some earlier runs swapped the optimiser.
# The models below are compiled with Adam, learning rate 0.001.


## Implementation 1 — rolling standardisation and prior sampling

This is the pipeline described in the report.

1. Pre-train for 50 epochs on sliding 90-day windows from 2005–2017.
2. Fine-tune on post-2017 blocks.
3. Decode 5,000 latent draws, keep the last day of each decoded window, and undo the z-score.
4. VaR is the 5th percentile of those scenarios. A violation is a simple return below that percentile.
5. AE ratio = violations / (5% of the length of the standardised test series).

Training plus the rolling fine-tune is a long run. The AE ratio is computed at the end of `fine_tune_and_calculate_var`; it is not hardcoded.


In [ ]:
class VariationalAutoencoder(tf.keras.Model):
    """VAE that reconstructs a window of standardised Nifty returns.

    Encoder: 90 -> 128 -> 64 -> (mu, log variance), latent width 32.
    Decoder: 32 -> 64 -> 128 -> 90.
    Loss: mean squared reconstruction error + beta * KL divergence to N(0, I).
    beta = 1 is the standard VAE objective used in the report.
    """

    def __init__(self, input_dim=90, latent_dim=32, beta=1.0):
        super(VariationalAutoencoder, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.beta = beta

        # Built as nested Keras models so the decoder can be called on its own
        # when we later draw scenarios from the latent space.
        self.encoder = self.build_encoder(input_dim)
        self.decoder = self.build_decoder(latent_dim)

        self.encoder.build(input_shape=(None, self.input_dim))
        self.decoder.build(input_shape=(None, self.latent_dim))

        self.compile(optimizer=Adam(learning_rate=0.001))

    def build_encoder(self, input_dim):
        inputs = Input(shape=(input_dim,))
        x = Dense(128, activation='relu')(inputs)
        x = Dense(64, activation='relu')(x)
        # Two heads: mean and log-variance of the approximate posterior q(z|x).
        z_mean = Dense(self.latent_dim, name='z_mean')(x)
        z_log_var = Dense(self.latent_dim, name='z_log_var')(x)

        def sampling(args):
            # Reparameterization trick: z = mu + sigma * epsilon, epsilon ~ N(0, I).
            # Sampling in this form lets gradients flow through mu and log-variance.
            z_mean, z_log_var = args
            epsilon = K.random_normal(shape=(K.shape(z_mean)[0], self.latent_dim))
            return z_mean + K.exp(0.5 * z_log_var) * epsilon

        z = Lambda(sampling, output_shape=(self.latent_dim,), name='z')([z_mean, z_log_var])
        return Model(inputs, [z_mean, z_log_var, z], name='encoder')

    def build_decoder(self, latent_dim):
        latent_inputs = Input(shape=(latent_dim,))
        x = Dense(64, activation='relu')(latent_inputs)
        x = Dense(128, activation='relu')(x)
        # Linear output: standardised returns are not bounded in (0, 1).
        decoder_outputs = Dense(self.input_dim)(x)
        return Model(latent_inputs, decoder_outputs, name='decoder')

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstructed = self.decoder(z)
        # Reconstruction term: how faithfully the window is rebuilt.
        reconstruction_loss = tf.reduce_mean(tf.keras.losses.mse(inputs, reconstructed))
        # KL term: keeps the latent posterior close to a standard normal,
        # which is what makes draws from N(0, I) valid scenarios at test time.
        kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
        self.add_loss(reconstruction_loss + self.beta * K.mean(kl_loss))
        return reconstructed


class EncodedVaR:
    """Train a VAE on Nifty windows and read 95% VaR from decoded scenarios.

    Default settings match the report: 90-day windows, 32-dimensional latent
    space, 95% confidence, beta = 1.
    """

    def __init__(self, window_size=90, latent_dim=32, confidence_level=0.95, beta=1.0, model_path="vae_model.weights.h5"):
        self.window_size = window_size
        self.latent_dim = latent_dim
        self.confidence_level = confidence_level
        self.beta = beta
        self.model_path = model_path
        self.vae = VariationalAutoencoder(input_dim=window_size, latent_dim=latent_dim, beta=beta)
        self.var_results = []

    def train_initial_model(self, data, epochs=50, batch_size=32):
        """Fit on 2005-2017 and save weights before any post-2017 fine-tuning."""
        processed_data = self.preprocess_data(data)
        processed_data = self.create_sliding_windows(processed_data)
        self.vae.build((None, self.window_size))
        self.vae.fit(processed_data, epochs=epochs, batch_size=batch_size, verbose=1)
        self.vae.save_weights(self.model_path)
        print(f"Model weights saved to {self.model_path}")

    def fine_tune_and_calculate_var(self, data, epochs=50, batch_size=16):
        """Walk the post-2017 sample, update the VAE, and score the next 10 returns.

        Each step fine-tunes on a short block, generates a VaR from the decoder,
        and counts how many of the following 10 simple returns fall below that VaR.
        The AE ratio is actual violations divided by the count implied by the
        confidence level (5% of the standardised series when confidence is 95%).
        The report's published AE ratio for this procedure is 1.04.
        """
        processed_data = self.preprocess_data(data)
        self.vae.load_weights(self.model_path)
        print("Pretrained model weights loaded.")

        violations = 0
        results = []

        # Step length is window_size + 10: a training block, then 10 scored days.
        for i in range(0, len(processed_data), self.window_size + 10):
            train_data = data[i:i + self.window_size + 2]
            train_data = self.preprocess_data(train_data)
            # The encoder expects rows of length window_size.
            train_data = train_data.reshape(-1, self.window_size)
            self.vae.fit(train_data, epochs=epochs, batch_size=batch_size, verbose=0)

            test_data = data[i + self.window_size:i + self.window_size + 10]
            test_data = test_data.pct_change().dropna()
            var = self.calculate_var(data[i:i + self.window_size])

            if i + self.window_size < len(data):
                actual_returns = data.pct_change().dropna().iloc[i + self.window_size:i + self.window_size + 10]
            else:
                actual_returns = np.nan

            # A violation is a realised simple return below the scenario VaR.
            if isinstance(test_data, (pd.Series, np.ndarray)):
                for ret in test_data:
                    violation = ret < var if not np.isnan(ret) else False
                    violations += int(violation)
                # The stored row keeps the last return of this 10-day block
                # together with the VaR that was compared with every day in it.
                results.append([var, ret, violation])
            else:
                results.append([var, np.nan, False])

        expected_violations = int((1 - self.confidence_level) * len(processed_data))
        ae_ratio = violations / expected_violations

        self.var_results = pd.DataFrame(results, columns=['VaR', 'Actual Return', 'Violation'])
        print(f"Total Violations: {violations}, Expected Violations: {expected_violations:.2f}, AE Ratio: {ae_ratio:.2f}")
        return self.var_results, ae_ratio

    def calculate_var(self, data, n_scenarios=5000):
        """5th percentile of decoded scenarios, mapped back to raw-return units."""
        scenarios = self.generate_scenarios(data, n_scenarios)
        returns = data.pct_change().dropna()
        current_mean = returns[-self.window_size:].mean()
        current_std = returns[-self.window_size:].std()
        # Undo the rolling standardisation so VaR is on the same scale as pct_change.
        destandardized_scenarios = scenarios * current_std + current_mean
        var = np.percentile(destandardized_scenarios, (1 - self.confidence_level) * 100)
        return var

    def generate_scenarios(self, data, n_scenarios):
        """Draw latent noise and decode it. Keep the last day of each window."""
        last_window = self.preprocess_data(data[-self.window_size - 1:])[-1:]
        # Scenarios come from the prior N(0, I), which the KL term trained against.
        # last_window is computed so the preprocessing path is the same as training;
        # the draws themselves are random latent points, not an encoding of that window.
        random_latent_points = tf.random.normal(shape=(n_scenarios, self.latent_dim))
        generated_samples = self.vae.decoder.predict(random_latent_points)
        return generated_samples[:, -1]

    def preprocess_data(self, data):
        """Simple returns, then a trailing 90-day z-score. 1e-8 avoids divide-by-zero."""
        returns = data.pct_change().dropna()
        rolling_mean = returns.rolling(window=self.window_size, min_periods=1).mean()
        rolling_std = returns.rolling(window=self.window_size, min_periods=1).std()
        return ((returns - rolling_mean) / (rolling_std + 1e-8)).dropna().values

    def create_sliding_windows(self, data):
        """One training row = 90 consecutive standardised returns."""
        return np.array([data[i:i + self.window_size] for i in range(len(data) - self.window_size + 1)])


def main():
    # Original Colab location, kept for traceability:
    # "/content/drive/MyDrive/MSDSM/Sem_4/FCP/EncodedVaR/Nifty50_data.xlsx"
    nifty_df = pd.read_excel(NIFTY_PATH)
    nifty_df['Date'] = pd.to_datetime(nifty_df['Dates'])
    # Report split: learn the latent distribution through 2017, then fine-tune after.
    train_df = nifty_df[(nifty_df['Date'] >= '2005-01-05') & (nifty_df['Date'] <= '2017-12-31')]
    test_df = nifty_df[nifty_df['Date'] > '2017-12-31']
    encoded_var = EncodedVaR()
    encoded_var.train_initial_model(train_df['NIFTY Index'])
    var_results, ae_ratio = encoded_var.fine_tune_and_calculate_var(test_df['NIFTY Index'])
    print("AE Ratio:", ae_ratio)
    return var_results


if __name__ == "__main__":
    var_results = main()


## Implementation 2 — posterior sampling and a 21-day step

Restart the kernel before running the next cell. It defines `VariationalAutoencoder` and `EncodedVaR` again.

Differences a reader should not miss:

- Inputs are raw simple returns, not rolling z-scores.
- Each pre-training step shows the network a single window (batch of one) and steps forward 21 days.
- Fine-tuning uses 10 epochs. Scenarios are drawn around the encoder mean and variance of the current window, not from a fresh N(0, I) that ignores the window.
- A test point counts as a violation only when it is **below the negative** of the predicted percentile.

`NIFTY_PATH` still comes from the setup cell, so run that cell again after a kernel restart.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K


class VariationalAutoencoder:
    """Second implementation: same 128-64-32 VAE, with an explicit train_step.

    Losses are returned separately (reconstruction, KL, total) so a training
    log can show whether the KL term is collapsing the latent space.
    """

    def __init__(self, input_dim=90, latent_dim=32, beta=1.0):
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.beta = beta
        self.build_vae()

    def build_vae(self):
        inputs = Input(shape=(self.input_dim,))
        x = Dense(128, activation='relu')(inputs)
        x = Dense(64, activation='relu')(x)
        z_mean = Dense(self.latent_dim, name="z_mean")(x)
        z_log_var = Dense(self.latent_dim, name="z_log_var")(x)

        def sampling(args):
            z_mean, z_log_var = args
            epsilon = K.random_normal(shape=(K.shape(z_mean)[0], self.latent_dim), mean=0., stddev=1.0)
            return z_mean + K.exp(0.5 * z_log_var) * epsilon

        z = Lambda(sampling, name="z")([z_mean, z_log_var])

        latent_inputs = Input(shape=(self.latent_dim,))
        x = Dense(64, activation='relu')(latent_inputs)
        x = Dense(128, activation='relu')(x)
        decoder_outputs = Dense(self.input_dim)(x)
        decoder = Model(latent_inputs, decoder_outputs, name="decoder")

        class VAE(keras.Model):
            def __init__(self, encoder, decoder, beta, **kwargs):
                super().__init__(**kwargs)
                self.encoder = encoder
                self.decoder = decoder
                self.beta = beta

            def train_step(self, data):
                with tf.GradientTape() as tape:
                    z_mean, z_log_var, z = self.encoder(data)
                    reconstruction = self.decoder(z)
                    reconstruction_loss = tf.reduce_mean(keras.losses.mse(data, reconstruction))
                    kl_loss = -0.5 * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
                    total_loss = reconstruction_loss + self.beta * kl_loss

                grads = tape.gradient(total_loss, self.trainable_weights)
                self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
                return {"loss": total_loss, "reconstruction_loss": reconstruction_loss, "kl_loss": kl_loss}

            def call(self, inputs):
                z_mean, z_log_var, z = self.encoder(inputs)
                reconstructed = self.decoder(z)
                return reconstructed

        self.encoder = Model(inputs, [z_mean, z_log_var, z], name="encoder")
        self.decoder = decoder
        self.vae = VAE(self.encoder, self.decoder, self.beta)
        self.vae.compile(optimizer=Adam(learning_rate=0.001))


class EncodedVaR:
    """Alternate backtest: 21-day steps, scenarios drawn around the encoded posterior.

    This class overwrites the EncodedVaR defined in the first implementation.
    Restart the kernel before running this cell if you still need the first model.
    """

    def __init__(self, window_size=90, latent_dim=32, confidence_level=0.95, beta=1.0, model_path="vae_model.weights.h5"):
        self.window_size = window_size
        self.latent_dim = latent_dim
        self.confidence_level = confidence_level
        self.beta = beta
        self.model_path = model_path
        self.vae = VariationalAutoencoder(input_dim=window_size, latent_dim=latent_dim, beta=beta)
        self.var_estimates = []
        self.step_size = 21  # about one trading month between refits

    def train_initial_model(self, data, epochs=50, batch_size=32):
        """Pre-train on raw simple-return windows, stepping 21 days at a time."""
        returns = data.pct_change().dropna()
        for i in range(0, len(returns) - self.window_size, self.step_size):
            train_data = returns[i:i + self.window_size].values.reshape(1, self.window_size)
            self.vae.vae.fit(train_data, epochs=epochs, batch_size=batch_size, verbose=0)

        self.vae.vae.save_weights(self.model_path)
        print(f"Model weights saved to {self.model_path}")

    def fine_tune_and_calculate_var(self, data, epochs=10, batch_size=16):
        """Fine-tune for 10 epochs per block and score the next `step_size` points.

        Violation rule in this version: a standardised test point is a breach
        when it is below the negative of the predicted percentile. That sign
        differs from the first implementation, which compares a raw simple
        return with a destandardised 5th percentile.
        """
        processed_data = self.preprocess_data(data)
        self.vae.vae.load_weights(self.model_path)
        print("Pretrained model weights loaded.")

        predicted_vars = []
        violations = 0
        test_window_size = len(processed_data) - self.window_size

        for i in range(0, test_window_size, self.window_size + self.step_size):
            train_data = processed_data[i:i + self.window_size]
            if len(train_data) < self.window_size:
                continue
            train_data = train_data.reshape(1, self.window_size)
            self.vae.vae.fit(train_data, epochs=epochs, batch_size=batch_size, verbose=0)

            # Encode the current window, then sample around that posterior.
            z_mean, z_log_var, _ = self.vae.encoder.predict(train_data)

            generated_scenarios = self.generate_scenarios(z_mean, z_log_var, 1000)
            var = np.percentile(generated_scenarios, (1 - self.confidence_level) * 100)
            predicted_vars.append(var)
            self.var_estimates.append(var)

            test_data = processed_data[i + self.window_size: i + self.window_size + self.step_size]
            for data_point in test_data:
                if data_point < -var:
                    violations += 1

        expected_violations = int((1 - self.confidence_level) * len(processed_data))
        ae_ratio = violations / expected_violations if expected_violations > 0 else np.nan

        print(f"Total Violations: {violations}, Expected Violations: {expected_violations}, AE Ratio: {ae_ratio:.2f}")
        return predicted_vars, ae_ratio

    def preprocess_data(self, data):
        """This version feeds raw simple returns to the network (no rolling z-score)."""
        returns = data.pct_change().dropna()
        return returns.values

    def generate_scenarios(self, z_mean, z_log_var, n_scenarios):
        """z = mu + sigma * epsilon, then decode. Keep the last day of each sample."""
        epsilon = np.random.normal(size=(n_scenarios, self.latent_dim))
        latent_samples = z_mean + np.exp(0.5 * z_log_var) * epsilon
        return self.vae.decoder.predict(latent_samples)[:, -1]


def main():
    nifty_df = pd.read_excel(NIFTY_PATH)
    nifty_df['Date'] = pd.to_datetime(nifty_df['Dates'])

    train_df = nifty_df[(nifty_df['Date'] >= '2005-01-05') & (nifty_df['Date'] <= '2017-12-31')]
    test_df = nifty_df[nifty_df['Date'] > '2017-12-31']

    encoded_var = EncodedVaR()
    encoded_var.train_initial_model(train_df['NIFTY Index'])
    predicted_vars, ae_ratio = encoded_var.fine_tune_and_calculate_var(test_df['NIFTY Index'])

    print("Predicted VaR:", predicted_vars)
    print("AE Ratio:", ae_ratio)


if __name__ == "__main__":
    main()
